# Data Analytics — intern_data_ikarus.csv
312 Amazon home & furniture products. This notebook covers missing values, price cleaning, category parsing, brand/color/material breakdowns, and word clouds.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast, re, warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 130
print('Ready')

In [ ]:
# Load raw CSV — price is '$xx.xx', categories is a Python list string
df_raw = pd.read_csv('../backend/data/intern_data_ikarus.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
# ── Missing value audit ─────────────────────────────────────────────
# description (49%), country_of_origin (60%), price (31%) are most missing
missing = df_raw.isnull().sum()
pct = (missing / len(df_raw) * 100).round(1)
print(pd.DataFrame({'count': missing, 'pct%': pct}))

fig, ax = plt.subplots(figsize=(10,4))
pct[pct>0].plot(kind='bar', ax=ax, color='#c4956a', edgecolor='white')
ax.set_title('Missing values by column (%)')
ax.set_ylabel('% missing')
plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# ── Clean price: remove $ → float ────────────────────────────────────
df = df_raw.copy()
df['price'] = df['price'].astype(str).str.replace('$','',regex=False).str.replace(',','',regex=False)
df['price'] = pd.to_numeric(df['price'], errors='coerce')
print(f'Price stats ({df["price"].count()} non-null):')
print(df['price'].describe().round(2))

In [ ]:
# ── Parse leaf category from list string ─────────────────────────────
# categories stored as: "['Home & Kitchen', 'Furniture', 'Chairs']"
def leaf_cat(s):
    try:
        lst = ast.literal_eval(str(s))
        return lst[-1].strip() if lst else 'Other'
    except:
        return str(s).split(',')[-1].strip().strip("'[] ") or 'Other'

df['leaf_category'] = df['categories'].apply(leaf_cat)
print('Top 12 leaf categories:')
print(df['leaf_category'].value_counts().head(12))

In [ ]:
# ── Category bar chart ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11,5))
df['leaf_category'].value_counts().head(15).plot(kind='barh', ax=ax, color='#8b5e3c')
ax.set_title('Top 15 Product Categories (leaf)')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

In [ ]:
# ── Price distribution ────────────────────────────────────────────────
p99 = df['price'].quantile(0.99)
price_c = df['price'].dropna()
price_c = price_c[price_c <= p99]

fig, (ax1, ax2) = plt.subplots(1,2, figsize=(13,4))
ax1.hist(price_c, bins=30, color='#8b5e3c', edgecolor='white', alpha=0.85)
ax1.set_title('Price Distribution'); ax1.set_xlabel('Price ($)')
ax2.boxplot(price_c, vert=False, patch_artist=True, boxprops=dict(facecolor='#d4b896'))
ax2.set_title('Price Box Plot'); ax2.set_xlabel('Price ($)')
plt.tight_layout(); plt.show()
print(f'Median ${price_c.median():.2f}  Mean ${price_c.mean():.2f}')

In [ ]:
# ── Brand analysis ────────────────────────────────────────────────────
brands = df['brand'].dropna().replace('', None).dropna().value_counts().head(15)
fig, ax = plt.subplots(figsize=(11,4))
brands.plot(kind='bar', ax=ax, color='#c4956a', edgecolor='white')
ax.set_title('Top 15 Brands')
plt.xticks(rotation=35, ha='right')
plt.tight_layout(); plt.show()
print(f'Unique brands: {df["brand"].nunique()}')

In [ ]:
# ── Color & Material ──────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1,2, figsize=(13,4))
df['color'].dropna().replace('',None).dropna().value_counts().head(12).plot(kind='bar', ax=ax1, color='#a0784d', edgecolor='white')
ax1.set_title('Top Colors'); ax1.tick_params(axis='x', rotation=35)
df['material'].dropna().replace('',None).dropna().value_counts().head(12).plot(kind='bar', ax=ax2, color='#d4b896', edgecolor='white')
ax2.set_title('Top Materials'); ax2.tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

In [ ]:
# ── Word Cloud — Product Titles ───────────────────────────────────────
try:
    from wordcloud import WordCloud
    text = ' '.join(df['title'].dropna())
    wc = WordCloud(width=900, height=360, background_color='white',
                   colormap='YlOrBr', max_words=100,
                   stopwords={'set','with','for','the','of','a','an','and','to','in','x'}).generate(text)
    plt.figure(figsize=(13,5))
    plt.imshow(wc, interpolation='bilinear'); plt.axis('off')
    plt.title('Word Cloud — Product Titles', fontsize=14)
    plt.tight_layout(); plt.show()
except ImportError:
    print('Install wordcloud: pip install wordcloud')

In [ ]:
# ── Price by Category ─────────────────────────────────────────────────
top8 = df['leaf_category'].value_counts().head(8).index.tolist()
pbc  = df[df['leaf_category'].isin(top8)][['leaf_category','price']].dropna()
fig, ax = plt.subplots(figsize=(12,5))
pbc.boxplot(column='price', by='leaf_category', ax=ax,
            boxprops=dict(color='#8b5e3c'), medianprops=dict(color='#c4956a', linewidth=2))
ax.set_title('Price by Category'); ax.set_xlabel('')
plt.suptitle(''); plt.xticks(rotation=25, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────
print('=== Dataset Summary ===')
print(f'Total products     : {len(df)}')
print(f'Unique brands      : {df["brand"].nunique()}')
print(f'Leaf categories    : {df["leaf_category"].nunique()}')
print(f'Products with price: {df["price"].notna().sum()}')
print(f'Avg price          : ${df["price"].mean():.2f}')
print(f'Products with image: {df["images"].notna().sum()}')